# Coffee17: EfficientNetV2 GAP vs HBP vs compact iSQRT-COV

Fail-fast validation seed 42. Checkpoint disimpan langsung ke Google Drive dan dapat dilanjutkan setelah runtime reset. Jangan membuka test pada tahap ini.

In [ ]:
# 1/3 SETUP REPO, GPU, DRIVE, DAN DATASET BERSIH
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
DATA = Path('/content/coffee17-covariance-data')
ORIGINAL = DATA / 'original'
CLEAN = DATA / 'clean'
FOLD = CLEAN / 'folds' / 'fold_1'
OUTPUT = Path('/content/drive/MyDrive/coffee17-compact-mpncov')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)

if not (ORIGINAL / 'source' / 'train').is_dir():
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17',
        '--output', str(ORIGINAL), '--archive', str(DATA / 'coffee17.zip'), '--seed', '42',
    ], check=True)
if not (FOLD / 'source' / 'val').is_dir():
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds',
        '--source-root', str(ORIGINAL / 'source'), '--output-root', str(CLEAN),
        '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1',
    ], check=True)
assert (FOLD / 'source' / 'val').is_dir(), f'Dataset belum siap: {FOLD}'
import torch
assert torch.cuda.is_available(), 'Aktifkan GPU T4: Runtime > Change runtime type > T4 GPU'
print('GPU    :', torch.cuda.get_device_name(0))
print('DATASET:', FOLD)
print('OUTPUT :', OUTPUT)

In [ ]:
# 2/3 JALANKAN FAIL-FAST SEED 42 (PROGRESS MUNCUL PER BATCH/EPOCH)
cmd = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_covariance_pooling_screening',
    '--data-root', str(FOLD), '--output-root', str(OUTPUT),
    '--seeds', '42', '--evaluation-split', 'val',
]
print('MENJALANKAN:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# 3/3 TAMPILKAN HASIL DAN PUTUSAN
decision_path = OUTPUT / 'val_reports' / 'covariance_failfast_val.json'
assert decision_path.is_file(), f'Hasil belum ditemukan: {decision_path}'
decision = json.loads(decision_path.read_text())
comparison = json.loads((OUTPUT / 'val_reports' / 'COV0_vs_COV2_aggregate.json').read_text())
print('=== COV0 GAP vs COV2 compact iSQRT-COV ===')
for key, label in [('macro_f1', 'Macro-F1'), ('hard_class_f1', 'Hard-F1'), ('worst_class_f1', 'Worst-F1')]:
    row = comparison['summary'][key]
    print(f"{label:9s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} ({row['delta_mean']:+.2%})")
print('KEPUTUSAN:', decision['decision'])
print('FAIL = refinement arsitektur dihentikan.')
print('PASS = baru konfirmasi validation seed 123 dan 2026; test tetap belum dibuka.')